In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.svm import SVR
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
from joblib import Parallel, delayed
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90 
df = pd.read_pickle("/planilhas/MFCC35.pkl")
print(f"Patients: {df['Patient_ID'].nunique()}")

____

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

Baseline - YA + A (Y = Delta_Y) MFCCs 1st

In [ ]:
df_model_YAA_S1 = df.copy()
df_model_YAA_S1 = df_model_YAA_S1.drop(columns=df_model_YAA_S1.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session8_Segment699_StdCoefficient13'].columns)
df_model_YAA_S1 = tv.standardized_delta_y(df_model_YAA_S1, MAX_HDRS, MAX_CDI)

In [ ]:
meta_YAA = ['Patient_ID', 'Age', 'Patient_Gender', 'Y_Standardized_Delta_Y']
df_model_YAA_S1 = mf.get_mfccs_per_segment_mean_std(df_model_YAA_S1, meta_YAA)
df_model_YAA_S1 = df_model_YAA_S1.dropna()

___

Running the experiments using LOO-CV patient-independent

In [ ]:
def baseline_dummy_leave_one_out(patient, df, patient_column, target_column):
    train_df = df[df[patient_column] != patient]
    test_df = df[df[patient_column] == patient]
    X_train = train_df.drop(columns=['Session', 'Segment', target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=['Session', 'Segment', target_column])
    y_test = test_df[target_column]

    dummy = DummyRegressor(strategy="mean")
    dummy.fit(X_train, y_train)
    y_pred = dummy.predict(X_test)
    return {
        "Model": "DummyRegressor(mean)",
        "Patient_ID": patient,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred)
    }

In [ ]:
def baseline_linear_clinical_leave_one_out(patient, df, patient_column, target_column, clinical_cols):
    for c in clinical_cols:
        if c not in df.columns:
            return None
    train_df = df[df[patient_column] != patient]
    test_df  = df[df[patient_column] == patient]
    X_train = train_df[clinical_cols]
    y_train = train_df[target_column].values
    X_test  = test_df[clinical_cols]
    y_test  = test_df[target_column].values
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)
    return {
        "Model": "LinearClinical(" + ",".join(clinical_cols) + ")",
        "Patient_ID": patient,
        "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "MSE": float(mean_squared_error(y_test, y_pred)),
        "MAE": float(mean_absolute_error(y_test, y_pred)),
    }

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"Processing baselines for Dataset {idx + 1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()
    # Dummy 
    results_dummy = Parallel(n_jobs=8)(
        delayed(baseline_dummy_leave_one_out)(p, df, 'Patient_ID', target_column)
        for p in unique_patients)
    # Linear using age and sex 
    clinical_cols = ["Age", "Patient_Gender"]
    results_lin = Parallel(n_jobs=8)(
        delayed(baseline_linear_clinical_leave_one_out)(p, df, 'Patient_ID', target_column, clinical_cols)
        for p in unique_patients
    )
    pack = [results_dummy, results_lin]
    current_results = [r for lst in pack for r in lst if r is not None]
    all_results.extend(current_results)
    df_current = pd.DataFrame(current_results)
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        summary_results.append({
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "RMSE_Mean": df_model["RMSE"].mean(),
            "RMSE_Std": df_model["RMSE"].std(),
            "MAE_Mean": df_model["MAE"].mean(),
        })
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)